In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import json

import cv2
import numpy as np
from google.colab.patches import cv2_imshow


# ── Input/output configuration ────────────────────────────────────────

IMAGE_PATH = Path("/content/drive/MyDrive/PPE Safety Compliance/FMDC1_00_20261012193338.jpg")
ANNOTATION_PATH = Path("/content/drive/MyDrive/PPE Safety Compliance/FMDC1_00_20261012193338.json")

OUTPUT_DIR = Path("/content/path_segmentation_image")
PATH_LABEL = "Pathway"

# Display only: saved results retain the input image resolution.
PREVIEW_MAX_WIDTH = 1100


# ── Load image and annotation ─────────────────────────────────────────

image = cv2.imread(str(IMAGE_PATH))

if image is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

if not ANNOTATION_PATH.exists():
    raise FileNotFoundError(f"Annotation not found: {ANNOTATION_PATH}")

annotation = json.loads(
    ANNOTATION_PATH.read_text(encoding="utf-8")
)

image_height, image_width = image.shape[:2]

annotation_width = int(annotation["imageWidth"])
annotation_height = int(annotation["imageHeight"])

if annotation_width <= 0 or annotation_height <= 0:
    raise ValueError("Invalid annotation dimensions.")

# This first test deliberately requires the original image dimensions.
if (image_width, image_height) != (
    annotation_width,
    annotation_height,
):
    raise ValueError(
        f"Image is {image_width} x {image_height}, but annotation is "
        f"{annotation_width} x {annotation_height}. "
        "Use the original annotated image for this first test."
    )


# ── Read pathway polygons ─────────────────────────────────────────────

shapes = [
    shape
    for shape in annotation.get("shapes", [])
    if shape.get("label") == PATH_LABEL
    and shape.get("shape_type") == "polygon"
]

if not shapes:
    raise ValueError(f"No polygon labelled '{PATH_LABEL}' found.")

polygons = []

for shape in shapes:
    points = np.asarray(shape["points"], dtype=np.float32)

    if (
        points.ndim != 2
        or points.shape[1] != 2
        or len(points) < 3
        or not np.isfinite(points).all()
    ):
        raise ValueError("Invalid pathway polygon coordinates.")

    polygon = np.rint(points).astype(np.int32).reshape(-1, 1, 2)
    polygons.append(polygon)


# ── Produce mask and isolated image ───────────────────────────────────

path_mask = np.zeros(
    (image_height, image_width),
    dtype=np.uint8,
)

# Fill separately so multiple pathway polygons form a union.
for polygon in polygons:
    cv2.fillPoly(path_mask, [polygon], 255)

isolated_path = cv2.bitwise_and(
    image,
    image,
    mask=path_mask,
)


# ── Produce overlay ──────────────────────────────────────────────────

overlay = image.copy()
inside_path = path_mask > 0

highlight_colour = np.array([0, 200, 0], dtype=np.float32)

overlay[inside_path] = np.clip(
    0.65 * image[inside_path].astype(np.float32)
    + 0.35 * highlight_colour,
    0,
    255,
).astype(np.uint8)

cv2.polylines(
    overlay,
    polygons,
    isClosed=True,
    color=(255, 255, 0),
    thickness=3,
    lineType=cv2.LINE_AA,
)


# ── Save full-resolution outputs ──────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outputs = {
    "path_mask.png": path_mask,
    "isolated_path.png": isolated_path,
    "path_overlay.png": overlay,
}

for filename, output_image in outputs.items():
    output_path = OUTPUT_DIR / filename

    if not cv2.imwrite(str(output_path), output_image):
        raise RuntimeError(f"Could not save: {output_path}")


# ── Display smaller previews ─────────────────────────────────────────

def show_preview(title, output_image):
    print(title)

    height, width = output_image.shape[:2]

    if width > PREVIEW_MAX_WIDTH:
        scale = PREVIEW_MAX_WIDTH / width
        output_image = cv2.resize(
            output_image,
            (
                PREVIEW_MAX_WIDTH,
                max(1, round(height * scale)),
            ),
            interpolation=cv2.INTER_NEAREST
            if output_image.ndim == 2
            else cv2.INTER_AREA,
        )

    cv2_imshow(output_image)


show_preview("Original image", image)
show_preview("Binary pathway mask", path_mask)
show_preview("Isolated pathway", isolated_path)
show_preview("Pathway overlay", overlay)

path_pixel_count = int(np.count_nonzero(path_mask))
path_fraction = path_pixel_count / path_mask.size

print(f"Image dimensions: {image_width} x {image_height}")
print(f"Pathway polygons: {len(polygons)}")
print(f"Pixels inside pathway: {path_pixel_count:,}")
print(f"Image area covered: {path_fraction:.2%}")
print(f"Saved outputs to: {OUTPUT_DIR}")

Output hidden; open in https://colab.research.google.com to view.